# S&P 500 Share Segmentation: Weekly Risk/Return and Adapted RFM Clustering



In [ ]:
# Colab: clone the repository to use the preserved data snapshot
# !git clone https://github.com/SenuthiW123/sp500-stock-segmentation.git   
# %cd sp500-stock-segmentation

In [ ]:
!pip install yfinance scikit-learn scipy matplotlib lxml  
import io, json, os, platform, sys, warnings
from datetime import date, datetime, timezone

import numpy as np
import pandas as pd
import requests
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import scipy
import sklearn
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

START, END = "2021-01-01", "2025-01-01"          # END is exclusive
BENCHMARK = "^GSPC"
SEED = 42
N_INIT = 50
K_RANGE = range(2, 9)
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..")) if "__file__" in globals() else os.getcwd()
DATA = os.path.join(ROOT, "data")
TAB = os.path.join(ROOT, "outputs", "tables")
FIG = os.path.join(ROOT, "outputs", "figures")
for d in (DATA, TAB, FIG):
    os.makedirs(d, exist_ok=True)
TICKER_FILE = os.path.join(DATA, "sp500_tickers.csv")
PRICE_FILE = os.path.join(DATA, "daily_prices.csv.gz")
META_FILE = os.path.join(DATA, "snapshot_metadata.json")

## 1.1 Ticker selection and data acquisition

In [ ]:
if os.path.exists(TICKER_FILE) and os.path.exists(PRICE_FILE):
    tickers = pd.read_csv(TICKER_FILE)
    prices = pd.read_csv(PRICE_FILE, parse_dates=["Date"])
    meta = json.load(open(META_FILE))
    print("Loaded cached snapshot retrieved", meta["retrieval_date_utc"])
else:
    import yfinance as yf
    if os.path.exists(TICKER_FILE):                   # reuse the stored constituent snapshot
        tickers = pd.read_csv(TICKER_FILE)
    else:
        html = requests.get(WIKI_URL, headers={"User-Agent": "Mozilla/5.0 (coursework data retrieval)"}, timeout=30).text
        tickers = pd.read_html(io.StringIO(html))[0]
        tickers["yahoo_symbol"] = tickers["Symbol"].str.replace(".", "-", regex=False)   # BRK.B -> BRK-B
        tickers.to_csv(TICKER_FILE, index=False)
    tickers_retrieved = datetime.fromtimestamp(os.path.getmtime(TICKER_FILE), timezone.utc).strftime("%Y-%m-%d %H:%M")
    import logging, time
    logging.getLogger("yfinance").setLevel(logging.CRITICAL)   # expected "no data" messages; missing symbols are reported below
    PART = os.path.join(DATA, "_partial")             # each symbol is written as it arrives, so an interrupted run resumes
    os.makedirs(PART, exist_ok=True)
    part_file = lambda s: os.path.join(PART, f"{s}.csv")
    pending = [s for s in tickers["yahoo_symbol"].tolist() + [BENCHMARK] if not os.path.exists(part_file(s))]
    for attempt in range(6):                          # retry symbols that time out or are rate-limited
        for i in range(0, len(pending), 25):
            batch = pending[i:i + 25]
            raw = yf.download(batch, start=START, end=END, auto_adjust=False, actions=True,
                              group_by="ticker", progress=False, threads=False)
            for s in batch:
                if s in raw.columns.get_level_values(0):
                    f = raw[s][raw[s]["Close"].notna()].reset_index()
                    if len(f):
                        f.assign(Ticker=s).to_csv(part_file(s), index=False)
            del raw
            time.sleep(1)
        still = [s for s in pending if not os.path.exists(part_file(s))]
        print(f"attempt {attempt + 1}: {len(os.listdir(PART))} symbols downloaded, {len(still)} pending", flush=True)
        if not still or (attempt and len(still) == len(pending)):   # stop once a retry recovers nothing
            break
        pending = still
        time.sleep(30)
    print("No Yahoo data for 2021-2024 (listed after the study period; excluded in 1.2):", still)
    prices = pd.concat([pd.read_csv(os.path.join(PART, f), parse_dates=["Date"]) for f in sorted(os.listdir(PART))],
                       ignore_index=True)
    prices.to_csv(PRICE_FILE, index=False, compression="gzip")
    meta = {"retrieval_date_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
            "tickers_retrieval_date_utc": tickers_retrieved,
            "source_tickers": WIKI_URL, "source_prices": f"Yahoo Finance via yfinance {yf.__version__}",
            "start": START, "end_exclusive": END, "auto_adjust": False}
    json.dump(meta, open(META_FILE, "w"), indent=2)

print(f"Constituents: {len(tickers)}  | price rows: {len(prices):,}  | symbols with data: {prices.Ticker.nunique()}")
versions = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
            "scikit-learn": sklearn.__version__, "scipy": scipy.__version__, "matplotlib": matplotlib.__version__}
try:
    import yfinance as yf
    versions["yfinance"] = yf.__version__
except ImportError:
    pass
print(versions)

## 1.2 Data validation and alignment

In [ ]:
bench = prices[prices.Ticker == BENCHMARK].set_index("Date").sort_index()
calendar = bench.index                       # benchmark trading calendar
first_day, last_day = calendar.min(), calendar.max()
log = []
valid = []
for _, row in tickers.iterrows():
    s = row["yahoo_symbol"]
    g = prices[(prices.Ticker == s) & prices["Close"].notna()]
    rec ={"Ticker": s, "Security": row["Security"], "Sector": row["GICS Sector"]}
    if g.empty:
        rec.update(status="excluded", reason="no data returned for study period"); log.append(rec); continue
    dup = int(g.Date.duplicated().sum())
    g = g.drop_duplicates("Date").set_index("Date").sort_index()
    rec.update(rows=len(g), first_date=g.index.min().date(), last_date=g.index.max().date(), duplicate_dates=dup,
               missing_close=int(g["Close"].isna().sum()), missing_adj=int(g["Adj Close"].isna().sum()),
               missing_volume=int(g["Volume"].isna().sum()),
               nonpositive_prices=int((g[["Open", "High", "Low", "Close", "Adj Close"]] <= 0).sum().sum()),
               volume_errors=int((g["Volume"] <= 0).sum()),
               missing_calendar_days=int(len(calendar.difference(g.index))),
               extra_days=int(len(g.index.difference(calendar))),
               splits=int((g["Stock Splits"] > 0).sum()), dividends=int((g["Dividends"] > 0).sum()))
    reasons = []
    if dup: reasons.append("duplicate dates")
    if rec["first_date"] > first_day.date() or rec["last_date"] < last_day.date():
        reasons.append("unequal history (listed/spun-off after start or delisted)")
    elif rec["missing_calendar_days"] or rec["missing_close"] or rec["missing_adj"] or rec["missing_volume"]:
        reasons.append("missing prices/volume inside study period")
    if rec["nonpositive_prices"]: reasons.append("nonpositive price")
    if rec["volume_errors"]: reasons.append("zero or negative volume")
    rec.update(status="excluded" if reasons else "included", reason="; ".join(reasons))
    log.append(rec)
    if not reasons:
        valid.append(s)
log = pd.DataFrame(log)
log.to_csv(os.path.join(TAB, "inclusion_exclusion_log.csv"), index=False)
print(log.status.value_counts())
print(log[log.status == "excluded"][["Ticker", "first_date", "last_date", "reason"]].to_string(index=False))

# Benchmark checks
print("Benchmark rows:", len(bench), "| duplicates:", int(bench.index.duplicated().sum()),
      "| missing Adj Close:", int(bench["Adj Close"].isna().sum()),
      "| nonpositive:", int((bench["Adj Close"] <= 0).sum()),
      "| Adj Close == Close on every day:", bool(np.allclose(bench["Adj Close"], bench["Close"])))

In [ ]:
# Corporate actions: confirm that split dates show no artificial jump in Adj Close (split-adjusted series)
wide = {c: prices[prices.Ticker.isin(valid)].pivot(index="Date", columns="Ticker", values=c).loc[calendar, valid]
        for c in ["Close", "Adj Close", "Volume", "Stock Splits", "Dividends"]}
adj, close, vol = wide["Adj Close"], wide["Close"], wide["Volume"]
ca = []
for s in valid:
    for d in wide["Stock Splits"].index[wide["Stock Splits"][s] > 0]:
        ca.append({"Ticker": s, "Date": d.date(), "split_ratio": wide["Stock Splits"].at[d, s],
                   "adj_close_return_on_split_day": adj[s].pct_change(fill_method=None).at[d],
                   "close_return_on_split_day": close[s].pct_change(fill_method=None).at[d]})
ca = pd.DataFrame(ca)
ca.to_csv(os.path.join(TAB, "corporate_actions_splits.csv"), index=False)
daily_adj_ret = adj.pct_change(fill_method=None)
jumps = daily_adj_ret[daily_adj_ret.abs() > 0.40].stack().dropna()   # pandas 3 stack keeps NaN
print(f"Splits in valid set: {len(ca)} | max |adj return| on split days: {ca.adj_close_return_on_split_day.abs().max():.3f}")
print(f"Shares paying dividends: {(wide['Dividends'] > 0).any().sum()} | daily |adj return| > 40%: {len(jumps)}")
print(jumps.to_string())

## 1.3 Weekly returns and weekly beta

In [ ]:
end_incl = pd.Timestamp(END) - pd.Timedelta(days=1)
def weekly(series_frame):
    w = series_frame.resample("W-FRI").last()          # final available trading price in each Friday-ending week
    monday = w.index - pd.Timedelta(days=4)
    complete = (monday >= pd.Timestamp(START)) & (w.index <= end_incl)   # drop weeks truncated by study bounds
    return w[complete]

wk_px = weekly(adj)
wk_bench = weekly(bench[["Adj Close"]])["Adj Close"]
assert wk_px.index.equals(wk_bench.index), "share and benchmark week labels differ"
trading_days_per_week = adj.iloc[:, 0].resample("W-FRI").count().reindex(wk_px.index)
print("Weekly price labels:", wk_px.index.min().date(), "to", wk_px.index.max().date(), "| weeks:", len(wk_px),
      "| holiday-shortened weeks retained:", int((trading_days_per_week < 5).sum()),
      "| weeks with no trading:", int((trading_days_per_week == 0).sum()))

wk_ret = wk_px.pct_change(fill_method=None).iloc[1:]   # P_t / P_{t-1} - 1, no filling
bm_ret = wk_bench.pct_change(fill_method=None).iloc[1:]
aligned = wk_ret.notna() & bm_ret.notna().values[:, None]
n_obs = aligned.sum()
mean_ret = wk_ret.mean()
beta = pd.Series({s: np.cov(wk_ret.loc[aligned[s], s], bm_ret[aligned[s]], ddof=1)[0, 1] /
                  np.var(bm_ret[aligned[s]], ddof=1) for s in valid})
print("Aligned weekly return observations per share:", n_obs.min(), "-", n_obs.max())
print(f"Benchmark mean weekly return {bm_ret.mean():.5f}, weekly sd {bm_ret.std():.5f}")
wk_ret.to_csv(os.path.join(TAB, "weekly_returns.csv"))

## 1.4 Adapted RFM

In [ ]:
obs_date = adj.dropna(how="any").index.max()      # final common completed trading date
assert obs_date == bench.index.max()
baseline = vol.shift(1).rolling(20, min_periods=20).mean()   # preceding 20 sessions, excludes test day
threshold = 2 * baseline
event = (vol > threshold) & threshold.notna()
window = vol.index[vol.index <= obs_date][-60:]
assert threshold.loc[window].notna().all().all(), "insufficient 20-session baseline inside 60-session window"
last_event = event.apply(lambda c: c.index[c].max() if c.any() else pd.NaT)
R = (obs_date - last_event).dt.days                      # calendar days; NaN if no event
F = event.loc[window].sum()
M = (close.loc[window] * vol.loc[window]).sum()
print("Observation date:", obs_date.date(), "| 60-session window:", window[0].date(), "to", window[-1].date())
print("Shares with no qualifying event (R missing):", int(R.isna().sum()))

In [ ]:
# Worked example (AAPL)
ex = "AAPL"
e_day = last_event[ex]
wx = pd.DataFrame({"Close": close.loc[window, ex], "Volume": vol.loc[window, ex],
                   "Baseline20": baseline.loc[window, ex], "Threshold": threshold.loc[window, ex],
                   "Event": event.loc[window, ex]})
wx["TradedValue"] = wx.Close * wx.Volume
wx.to_csv(os.path.join(TAB, f"worked_example_{ex}.csv"))
print(f"{ex}: last event {e_day.date()} volume {vol.at[e_day, ex]:,.0f} > threshold {threshold.at[e_day, ex]:,.0f};"
      f" R = {R[ex]:.0f} days, F = {F[ex]}, M = ${M[ex]/1e9:,.2f}bn")
print(wx[wx.Event])

## 1.5 Feature table and summaries

In [ ]:
info = log.set_index("Ticker").loc[valid, ["Security", "Sector"]]
feat = info.assign(mean_weekly_return=mean_ret, weekly_beta=beta, n_weekly_obs=n_obs,
                   last_event_date=last_event.dt.date, R=R, F=F, M=M, log1p_M=np.log1p(M))
feat["rfm_valid"] = feat.R.notna()
common = feat.index[feat.rfm_valid]                  # common valid share set for all clustering
print("Common valid share set:", len(common))
summary = feat[["mean_weekly_return", "weekly_beta", "R", "F", "M", "log1p_M"]].describe().T
summary["skew"] = feat[summary.index].skew()
summary.to_csv(os.path.join(TAB, "feature_summary.csv"))
print(summary.round(4))

fig, ax = plt.subplots(2, 3, figsize=(13, 7))
for a, c in zip(ax.flat, ["mean_weekly_return", "weekly_beta", "R", "F", "M", "log1p_M"]):
    a.hist(feat.loc[common, c], bins=40, color="#3b6ea5")
    a.set_title(f"{c} (skew {feat.loc[common, c].skew():.2f})")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig01_feature_distributions.png"), dpi=150); plt.close()

## 2.1 Feature preparation

In [ ]:
X_rr_raw = feat.loc[common, ["mean_weekly_return", "weekly_beta"]]
X_rfm_raw = feat.loc[common, ["R", "F", "log1p_M"]]
def iqr_outliers(df):
    q1, q3 = df.quantile(.25), df.quantile(.75)
    return ((df < q1 - 1.5 * (q3 - q1)) | (df > q3 + 1.5 * (q3 - q1))).sum()
print("IQR outliers:\n", pd.concat([iqr_outliers(X_rr_raw), iqr_outliers(feat.loc[common, ["R", "F", "M", "log1p_M"]])]))
X_rr = pd.DataFrame(StandardScaler().fit_transform(X_rr_raw), index=common, columns=X_rr_raw.columns)
X_rfm = pd.DataFrame(StandardScaler().fit_transform(X_rfm_raw), index=common, columns=X_rfm_raw.columns)
X_rr.to_csv(os.path.join(TAB, "matrix_risk_return_standardised.csv"))
X_rfm.to_csv(os.path.join(TAB, "matrix_rfm_standardised.csv"))

## 2.2-2.3 Ward agglomerative and K-means: cluster count evidence

In [ ]:
def evaluate(X, name):
    rows = []
    for k in K_RANGE:
        w = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)
        km = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit(X)
        rows.append({"matrix": name, "k": k,
                     "ward_silhouette": silhouette_score(X, w), "ward_sizes": sorted(np.bincount(w), reverse=True),
                     "kmeans_silhouette": silhouette_score(X, km.labels_), "kmeans_inertia": km.inertia_,
                     "kmeans_sizes": sorted(np.bincount(km.labels_), reverse=True)})
    return pd.DataFrame(rows)

ev = pd.concat([evaluate(X_rr, "risk_return"), evaluate(X_rfm, "rfm")], ignore_index=True)
ev.to_csv(os.path.join(TAB, "k_selection_evidence.csv"), index=False)
print(ev.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for name, mk in [("risk_return", "o"), ("rfm", "s")]:
    e = ev[ev.matrix == name]
    ax[0].plot(e.k, e.ward_silhouette, marker=mk, label=name); ax[1].plot(e.k, e.kmeans_silhouette, marker=mk, label=name)
    ax[2].plot(e.k, e.kmeans_inertia, marker=mk, label=name)
for a, t in zip(ax, ["Ward silhouette", "K-means silhouette", "K-means inertia (elbow)"]):
    a.set_title(t); a.set_xlabel("k"); a.legend()
plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig02_k_selection.png"), dpi=150); plt.close()

for X, name, tag in [(X_rr, "Risk/return matrix", "risk_return"), (X_rfm, "RFM matrix", "rfm")]:
    Z = linkage(X, method="ward", metric="euclidean")
    plt.figure(figsize=(12, 5))
    dendrogram(Z, truncate_mode="lastp", p=40, show_leaf_counts=True, leaf_rotation=90, color_threshold=0)
    plt.title(f"Ward dendrogram - {name} (standardised, last 40 merges; leaf counts in brackets)")
    plt.ylabel("Ward linkage distance")
    plt.tight_layout(); plt.savefig(os.path.join(FIG, f"fig03_dendrogram_{tag}.png"), dpi=150); plt.close()

In [ ]:
# Selected cluster counts. Rule: highest silhouette among k whose smallest cluster holds >= 5% of shares,
# then checked for interpretable profiles (see report). Same k is used for both algorithms on a matrix.
MIN_SHARE = 0.05
def select_k(e, algo):
    ok = e[e[f"{algo}_sizes"].apply(min) >= MIN_SHARE * len(common)]
    return int(ok.loc[ok[f"{algo}_silhouette"].idxmax(), "k"])
K_SEL = {}
for name in ["risk_return", "rfm"]:
    e = ev[ev.matrix == name]
    K_SEL[name] = {"ward": select_k(e, "ward"), "kmeans": select_k(e, "kmeans")}
print("Selected k:", K_SEL)

## Baseline solutions, profiles and comparison (2.4)

In [ ]:
def relabel(labels, order_by):
    # renumber clusters 1..k by ascending mean of `order_by` so labels are readable
    m = pd.Series(order_by.values).groupby(labels).mean().sort_values()
    return pd.Series(labels).map({old: new + 1 for new, old in enumerate(m.index)}).values

sol = pd.DataFrame(index=common)
for X, name, key in [(X_rr, "rr", "risk_return"), (X_rfm, "rfm", "rfm")]:
    order = X_rr_raw.weekly_beta if name == "rr" else X_rfm_raw.log1p_M
    sol[f"{name}_ward"] = relabel(AgglomerativeClustering(n_clusters=K_SEL[key]["ward"], linkage="ward").fit_predict(X), order)
    sol[f"{name}_kmeans"] = relabel(KMeans(n_clusters=K_SEL[key]["kmeans"], n_init=N_INIT, random_state=SEED).fit_predict(X), order)

orig = feat.loc[common, ["mean_weekly_return", "weekly_beta", "R", "F", "M"]].copy()
orig["mean_weekly_return_pct"] = orig.pop("mean_weekly_return") * 100
orig["M_usd_bn"] = orig.pop("M") / 1e9
profiles = []
for c in sol.columns:
    cols = ["mean_weekly_return_pct", "weekly_beta"] if c.startswith("rr") else ["R", "F", "M_usd_bn"]
    p = orig.groupby(sol[c])[cols].agg(["mean", "median"])
    p.columns = [f"{a}_{b}" for a, b in p.columns]
    p.insert(0, "n", sol[c].value_counts().sort_index())
    p.insert(0, "solution", c)
    profiles.append(p.reset_index(names="cluster"))
profiles = pd.concat(profiles, ignore_index=True)
profiles.to_csv(os.path.join(TAB, "cluster_profiles_original_units.csv"), index=False)
print(profiles.round(3).to_string(index=False))

ari = pd.DataFrame([
    {"comparison": "Risk/return: Ward vs K-means", "ARI": adjusted_rand_score(sol.rr_ward, sol.rr_kmeans)},
    {"comparison": "RFM: Ward vs K-means", "ARI": adjusted_rand_score(sol.rfm_ward, sol.rfm_kmeans)},
    {"comparison": "Ward risk/return vs Ward RFM", "ARI": adjusted_rand_score(sol.rr_ward, sol.rfm_ward)}])
print(pd.crosstab(sol.rr_ward, sol.rr_kmeans, rownames=["RR Ward"], colnames=["RR K-means"]))
print(pd.crosstab(sol.rfm_ward, sol.rfm_kmeans, rownames=["RFM Ward"], colnames=["RFM K-means"]))

In [ ]:
cmap = plt.get_cmap("tab10")
fig, ax = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for a, c, t in zip(ax, ["rr_ward", "rr_kmeans"], ["Ward", "K-means"]):
    for k in sorted(sol[c].unique()):
        m = sol[c] == k
        a.scatter(orig.weekly_beta[m], orig.mean_weekly_return_pct[m], s=12, color=cmap(k - 1), label=f"Cluster {k} (n={m.sum()})")
    a.set_title(f"Risk/return clusters - {t}"); a.set_xlabel("Weekly beta vs ^GSPC"); a.legend(fontsize=8)
ax[0].set_ylabel("Mean weekly return (%)")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig04_risk_return_clusters.png"), dpi=150); plt.close()

pca = PCA(n_components=2, random_state=SEED).fit(X_rfm)
pc = pca.transform(X_rfm)
fig, ax = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for a, c, t in zip(ax, ["rfm_ward", "rfm_kmeans"], ["Ward", "K-means"]):
    for k in sorted(sol[c].unique()):
        m = (sol[c] == k).values
        a.scatter(pc[m, 0], pc[m, 1], s=12, color=cmap(k - 1), label=f"Cluster {k} (n={m.sum()})")
    a.set_title(f"RFM clusters - {t}"); a.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} of variance)")
    a.legend(fontsize=8)
ax[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} of variance)")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig05_rfm_clusters_pca.png"), dpi=150); plt.close()
loadings = pd.DataFrame(pca.components_.T, index=X_rfm.columns, columns=["PC1", "PC2"])
loadings.to_csv(os.path.join(TAB, "rfm_pca_loadings.csv"))
print("PCA explained variance:", pca.explained_variance_ratio_.round(3), "\n", loadings.round(3))

## 2.5 RFM sensitivity: correlations and removal of Monetary

In [ ]:
corr = pd.concat({"pearson": X_rfm_raw.corr(), "spearman": X_rfm_raw.corr(method="spearman")})
corr.to_csv(os.path.join(TAB, "rfm_correlations.csv"))
print(corr.round(3))
plt.figure(figsize=(4.5, 3.8))
plt.imshow(X_rfm_raw.corr(), cmap="RdBu_r", vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(3), X_rfm_raw.columns); plt.yticks(range(3), X_rfm_raw.columns)
for i in range(3):
    for j in range(3):
        plt.text(j, i, f"{X_rfm_raw.corr().iat[i, j]:.2f}", ha="center", va="center")
plt.title("Pearson correlation: R, F, log1p(M)")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig06_rfm_correlation.png"), dpi=150); plt.close()

X_rf = StandardScaler().fit_transform(feat.loc[common, ["R", "F"]])          # re-standardised afresh
sol["rf_ward"] = relabel(AgglomerativeClustering(n_clusters=K_SEL["rfm"]["ward"], linkage="ward").fit_predict(X_rf),
                         X_rfm_raw.log1p_M)
ari = pd.concat([ari, pd.DataFrame([{"comparison": "Ward RFM vs Ward RF (M removed)",
                                     "ARI": adjusted_rand_score(sol.rfm_ward, sol.rf_ward)}])], ignore_index=True)
# Robustness (treatment of skewed R): R is kept in calendar days for interpretability; check log1p(R) instead
X_logr = StandardScaler().fit_transform(np.column_stack([np.log1p(feat.loc[common, "R"]), X_rfm_raw[["F", "log1p_M"]]]))
logr_ward = AgglomerativeClustering(n_clusters=K_SEL["rfm"]["ward"], linkage="ward").fit_predict(X_logr)
ari = pd.concat([ari, pd.DataFrame([{"comparison": "Ward RFM vs Ward RFM with log1p(R) (robustness)",
                                     "ARI": adjusted_rand_score(sol.rfm_ward, logr_ward)}])], ignore_index=True)
ari.to_csv(os.path.join(TAB, "ari_summary.csv"), index=False)
print(ari.round(3))
print(pd.crosstab(sol.rfm_ward, sol.rf_ward, rownames=["Ward RFM"], colnames=["Ward RF"]))
tv = pd.concat({c: orig.groupby(sol[c])[["R", "F", "M_usd_bn"]].median().assign(n=sol[c].value_counts())
                for c in ["rfm_ward", "rf_ward"]})
tv.to_csv(os.path.join(TAB, "monetary_removal_traded_value_profiles.csv"))
print(tv.round(2))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for a, c, t in zip(ax, ["rfm_ward", "rf_ward"], ["Ward on R, F, log1p(M)", "Ward on R, F only"]):
    data = [orig.M_usd_bn[sol[c] == k] for k in sorted(sol[c].unique())]
    a.boxplot(data, tick_labels=[f"C{k}" for k in sorted(sol[c].unique())], showfliers=False)
    a.set_yscale("log"); a.set_title(t); a.set_xlabel("Cluster")
ax[0].set_ylabel("60-session traded value M (USD bn, log scale)")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig07_monetary_removal.png"), dpi=150); plt.close()

## 3.2 Sector composition and outputs

In [ ]:
sector = feat.loc[common, "Sector"]
comp = []
for c in ["rr_ward", "rfm_ward"]:
    t = pd.crosstab(sector, sol[c], normalize="columns") * 100
    t.columns = [f"{c}_C{k}" for k in t.columns]
    comp.append(t)
comp = pd.concat(comp, axis=1)
comp.to_csv(os.path.join(TAB, "sector_composition_pct.csv"))
print(comp.round(1))
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
colors = plt.get_cmap("tab20").colors
for a, c, t in zip(ax, ["rr_ward", "rfm_ward"], ["Risk/return clusters (Ward)", "RFM clusters (Ward)"]):
    ct = pd.crosstab(sol[c], sector, normalize="index") * 100
    ct.plot(kind="bar", stacked=True, ax=a, color=colors[:ct.shape[1]], legend=False, width=0.75)
    a.set_title(t); a.set_xlabel("Cluster"); a.set_xticklabels([f"C{k} (n={(sol[c]==k).sum()})" for k in ct.index], rotation=0)
ax[0].set_ylabel("% of cluster")
ax[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig08_sector_composition.png"), dpi=150); plt.close()
print(pd.crosstab(sol.rr_ward, sol.rfm_ward, rownames=["RR Ward"], colnames=["RFM Ward"]))

feature_table = feat.join(sol)
feature_table.index.name = "Ticker"
feature_table.to_csv(os.path.join(TAB, "feature_table_with_clusters.csv"))
run_meta = {**meta, "observation_date": str(obs_date.date()), "rfm_window": [str(window[0].date()), str(window[-1].date())],
            "weekly_labels": [str(wk_ret.index.min().date()), str(wk_ret.index.max().date())],
            "aligned_weekly_obs": int(n_obs.max()), "n_constituents": int(len(tickers)), "n_included": len(valid),
            "n_common_set": int(len(common)), "selected_k": K_SEL, "seed": SEED, "n_init": N_INIT,
            "package_versions": versions}
json.dump(run_meta, open(os.path.join(TAB, "run_metadata.json"), "w"), indent=2, default=str)
print(json.dumps(run_meta, indent=2, default=str))